# 02. Loading Documents: Text, Markdown, and Real PDFs

> **Promise:** By the end of this notebook, you will be able to load, inspect, clean, and attach metadata to raw text, Markdown, and PDF documents using modern LangChain loaders.

### Learning Objectives
- Understand the anatomy of a LangChain Document object (page_content string and metadata dictionary).
- Load .txt, .md, and real PDF files using TextLoader, PyPDFLoader, and directory scanners.
- Clean and standardize document metadata to power downstream search filters and citations.

### Prerequisites
- Notebook 01 (RAG The Big Picture)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.helpers import get_data_dir

data_dir = get_data_dir()
print(f"Data directory: {data_dir}")
print("Files available:", [f.name for f in data_dir.iterdir() if f.is_file()])

Data directory: C:\project 03\llm\course\data
Files available: ['company_handbook.txt', 'eval_questions.json', 'generate_pdf.py', 'policy_brief.pdf', 'product_faq.md', 'recent_changelog.md']


## 2. Concept & Intuition

### The Fundamental Unit: The `Document` Object

In LangChain, all textual data flows through a single standardized dataclass called a **Document**.

A Document has exactly two fields:
1. `page_content: str` — The actual text (the raw knowledge).
2. `metadata: dict` — A dictionary of key-value pairs describing where the text came from (file path, page number, creation date, author, department).

#### Why Metadata Matters
Imagine a library with 10,000 loose sheets of paper. Without titles or page numbers, finding anything or verifying a fact is impossible. Metadata is the index card attached to each sheet.

```
┌────────────────────────────────────────────────────────┐
│ Document Object                                        │
├────────────────────────────────────────────────────────┤
│ metadata: {                                            │
│    "source": "company_handbook.txt",                   │
│    "page": 1,                                          │
│    "category": "policy"                                │
│ }                                                      │
├────────────────────────────────────────────────────────┤
│ page_content:                                          │
│ "Employees traveling on company business are           │
│  reimbursed up to $150 per day for meals..."           │
└────────────────────────────────────────────────────────┘
```

## 3. Step-by-Step Hands-On Build

### Step 1: Creating a Document Object by Hand
Let's import `Document` from `langchain_core.documents` and construct one manually to see how simple it is.

In [2]:
from langchain_core.documents import Document

manual_doc = Document(
    page_content="Novasphere provides 24/7 mission-critical support on the Enterprise plan.",
    metadata={"source": "manual_entry", "department": "Customer Engineering"}
)

print("Type:", type(manual_doc))
print("Content:", manual_doc.page_content)
print("Metadata:", manual_doc.metadata)

Type: <class 'langchain_core.documents.base.Document'>
Content: Novasphere provides 24/7 mission-critical support on the Enterprise plan.
Metadata: {'source': 'manual_entry', 'department': 'Customer Engineering'}


> **What just happened?**
> Every document loader in LangChain produces objects following this exact structure.

### Step 2: Loading Text and Markdown Files
Now let's use `TextLoader` from `langchain_community.document_loaders` to load `company_handbook.txt` and `product_faq.md`.

In [3]:
from langchain_community.document_loaders import TextLoader

handbook_loader = TextLoader(str(data_dir / "company_handbook.txt"), encoding="utf-8")
handbook_docs = handbook_loader.load()

faq_loader = TextLoader(str(data_dir / "product_faq.md"), encoding="utf-8")
faq_docs = faq_loader.load()

print(f"Handbook docs: {len(handbook_docs)}, chars: {len(handbook_docs[0].page_content)}")
print("Handbook Metadata:", handbook_docs[0].metadata)
print(f"FAQ docs: {len(faq_docs)}, chars: {len(faq_docs[0].page_content)}")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_24692\4046738727.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Handbook docs: 1, chars: 4163
Handbook Metadata: {'source': 'C:\\project 03\\llm\\course\\data\\company_handbook.txt'}
FAQ docs: 1, chars: 2973


> **What just happened?**
> Notice that TextLoader loads the entire file into a single Document object and attaches the file path to metadata['source'].

### Step 3: Loading Real PDF Files with PyPDFLoader
PDFs are multi-page binary documents. `PyPDFLoader` automatically parses page streams and produces one Document object per page.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = data_dir / "policy_brief.pdf"
pdf_loader = PyPDFLoader(str(pdf_path))
pdf_docs = pdf_loader.load()

print(f"Loaded {len(pdf_docs)} page(s) from {pdf_path.name}")
for idx, page_doc in enumerate(pdf_docs):
    print(f"\n--- Page {idx+1} (Source: {Path(page_doc.metadata.get('source', '')).name}, Page: {page_doc.metadata.get('page')}) ---")
    print("Content preview:", page_doc.page_content[:180].strip())

Loaded 1 page(s) from policy_brief.pdf

--- Page 1 (Source: policy_brief.pdf, Page: 0) ---
Content preview: Novasphere Dynamics - Autonomous Robotics Safety &
Deployment Brief
Document ID: SAF-2026-B | Classification: Confidential Internal
1. Purpose & Scope
This technical brief specifie


> **What just happened?**
> PyPDFLoader automatically attached the 'page' number to the metadata! This enables page-level citations in user interfaces.

### Step 4: Loading an Entire Directory of Mixed Files
In production, you want to point your pipeline at a folder of mixed `.txt`, `.md`, and `.pdf` files. Let's write a robust loader loop.

In [5]:
all_docs = []

for file_path in data_dir.iterdir():
    if file_path.suffix.lower() == ".pdf":
        all_docs.extend(PyPDFLoader(str(file_path)).load())
    elif file_path.suffix.lower() in [".txt", ".md"]:
        all_docs.extend(TextLoader(str(file_path), encoding="utf-8").load())

print(f"Total documents loaded from directory: {len(all_docs)}")
for d in all_docs:
    print(f"  * {Path(d.metadata['source']).name} ({len(d.page_content)} characters)")

Total documents loaded from directory: 4
  * company_handbook.txt (4163 characters)
  * policy_brief.pdf (1334 characters)
  * product_faq.md (2973 characters)
  * recent_changelog.md (2118 characters)


> **What just happened?**
> We now have a clean collection of Document objects ready for splitting and indexing.

### Step 5: Cleaning and Normalizing Metadata
Raw metadata often contains long absolute filesystem paths (`C:\project 03\...`). Let's normalize metadata for readability and auditability.

In [6]:
for doc in all_docs:
    full_source = doc.metadata.get("source", "")
    doc.metadata["filename"] = Path(full_source).name
    doc.metadata["char_count"] = len(doc.page_content)
    if "handbook" in doc.metadata["filename"]:
        doc.metadata["category"] = "policy"
    elif "faq" in doc.metadata["filename"]:
        doc.metadata["category"] = "product"
    elif "changelog" in doc.metadata["filename"]:
        doc.metadata["category"] = "releases"
    else:
        doc.metadata["category"] = "general"

print("Sample Cleaned Metadata:")
print(all_docs[0].metadata)

Sample Cleaned Metadata:
{'source': 'C:\\project 03\\llm\\course\\data\\company_handbook.txt', 'filename': 'company_handbook.txt', 'char_count': 4163, 'category': 'policy'}


> **What just happened?**
> Our documents now have clean, consistent metadata categories for search filtering!

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Empty Document Guard):** Write a list comprehension that filters out any documents that have fewer than 20 characters of `page_content`.
2. **Exercise 2 (Web Loader with Offline Fallback):** Write a minimal function that attempts to load text from a URL using `requests` and `BeautifulSoup`, falling back to a local sample string if offline.

In [7]:
non_empty_docs = [d for d in all_docs if len(d.page_content.strip()) >= 20]
print(f"Retained {len(non_empty_docs)} of {len(all_docs)} documents.")

import requests
from bs4 import BeautifulSoup

def safe_load_web(url: str) -> Document:
    try:
        resp = requests.get(url, timeout=3.0)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        return Document(page_content=soup.get_text()[:500], metadata={"source": url})
    except Exception as e:
        print(f"[Offline Fallback] Network request failed ({e}). Using cached document.")
        return Document(page_content="Cached Web Content: Novasphere Robotics Documentation", metadata={"source": url, "cached": True})

sample_web_doc = safe_load_web("https://example.com/novasphere-docs")
print("Web Doc Source:", sample_web_doc.metadata["source"])

Retained 4 of 4 documents.


[Offline Fallback] Network request failed (404 Client Error: Not Found for url: https://example.com/novasphere-docs). Using cached document.
Web Doc Source: https://example.com/novasphere-docs


## 5. Common Mistakes & Pitfalls
- **Ignoring File Encodings:** Always specify `encoding='utf-8'` in TextLoader on Windows, or UTF-8 characters will raise UnicodeDecodeError.
- **Overwriting Metadata:** When adding custom fields, do not overwrite existing keys like `source` or `page`.
- **Loading Scanned PDFs Without OCR:** Standard PDF loaders only extract embedded digital text. Scanned images will return empty strings.
- **Discarding Document Objects:** Don't convert Documents to raw strings too early; you will lose the metadata needed for citations.

---

## 6. Notebook Recap
- The Document object (`page_content` + `metadata`) is the universal data unit in LangChain.
- TextLoader loads plain text and Markdown; PyPDFLoader extracts text and records page numbers.
- Clean metadata (filename, page, category) is essential for filtering and source citations.
- Always handle character encodings and empty pages defensively.


## 7. What's Next?

In **03_splitting_text.ipynb**, we take our loaded documents and solve the chunking puzzle: how large should chunks be, what is chunk overlap, and how do we prevent answers from being cut in half?

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The LangChain Document Object:** Understanding page_content and metadata.
- **02:30 - Loading Text and Markdown:** TextLoader and UTF-8 encoding best practices.
- **05:10 - Parsing PDFs with PyPDFLoader:** Extracting page numbers and handling multi-page documents.
- **08:20 - Directory Loading and File Iteration:** Loading heterogeneous folders of company files.
- **11:00 - Metadata Cleaning and Normalization:** Preparing metadata for downstream vector filtering.
